In [1]:
import pysam
import numpy as np
import pandas as pd
import itertools
import re


from Bio import SeqIO
from Bio.Seq import Seq
from tqdm import tqdm
from collections import defaultdict

In [2]:
def get_align_flag(align, mapq_thre =20):
    is_paired  = align.is_proper_pair
    is_mapped  = (not align.is_unmapped)
    is_primary = (not align.is_secondary) and (not align.is_supplementary)
    is_qualified = (not align.is_qcfail) and align.mapq >= mapq_thre
    is_fully_align = all(op in (0, 7, 8) for op, length in align.cigar)
    return is_paired and is_mapped and is_primary and is_qualified and is_fully_align


def read_pair_generator(bam_file, region_string=None):
    """
    Generate read pairs in a BAM file or within a region string.
    Reads are added to read_dict until a pair is found.
    """
    read_dict = defaultdict(lambda: [None, None])
    bam = pysam.AlignmentFile(bam_file, "rb")
    #bam.reset()
    
    for align in bam.fetch(until_eof=True, region=region_string):        
        if not get_align_flag(align):
            continue
        
        qname = align.query_name
        if qname not in read_dict.keys():
            read_dict[qname][int(align.is_read2)] = align
        else:
            read_dict[qname][int(align.is_read2)] = align
            yield read_dict[qname]
            del read_dict[qname]

In [3]:
# problematic, the itertools.groupby doesn't work as expected
# def gen_read_pair(bam_file, contig_id):
#     bam = pysam.AlignmentFile(bam_file, "rb")
    
#     for _, reads in itertools.groupby(bam.fetch(contig=contig_id), key=lambda x: x.query_name):
#         reads = list(reads)
#         if len(reads) == 2:
#             read1, read2 = reads
#             read1_flag = get_align_flag(read1)
#             read2_flag = get_align_flag(read2)

#             if read1_flag and read2_flag and read1.reference_name == read2.reference_name:
#                 if read1.is_read1 and read2.is_read2:
#                     yield read1, read2
#                 else:
#                     yield read2, read1

# problematic function test
# read1_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}
# read2_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}
# working_path = "/home/wbguo/iproject/BSReadSim/test/"
# bam_file  = working_path + "/data/sim/pe_d/sim.mkdup.sorted.bam"
# for read1, read2 in gen_read_pair(bam_file, 'chr21'):
#     read1_tags[read1.get_tag('YS')] += 1
#     read2_tags[read2.get_tag('YS')] += 1

In [4]:
## index by name, takes ~85Gb for the WGBS, not suitable
# working_path = "/home/wbguo/iproject/BSReadSim/test/"
# bam_file  = working_path + "/data/WGBS/ERR2359938.mkdup.sorted.bam"
# bam = pysam.AlignmentFile(bam_file, "rb")
# x = pysam.IndexedReads(bam)
# x.build()

# WGBS

In [5]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
bam_file  = working_path + "/data/WGBS/ERR2359938.mkdup.sorted.bam"

In [6]:
read1_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}
read2_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}

for read1, read2 in read_pair_generator(bam_file, 'chr21'):
    read1_tags[read1.get_tag('YS')] += 1
    read2_tags[read2.get_tag('YS')] += 1

In [7]:
read1_tags

{'C_C2T': 946362, 'W_C2T': 927948, 'W_G2A': 0, 'C_G2A': 0}

In [8]:
read2_tags

{'C_C2T': 0, 'W_C2T': 0, 'W_G2A': 927948, 'C_G2A': 946362}

# RRBS

In [9]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
bam_file  = working_path + "/data/RRBS/SRR6294859.sorted.bam"

In [10]:
read1_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}
read2_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}

for read1, read2 in read_pair_generator(bam_file, 'chr21'):
    read1_tags[read1.get_tag('YS')] += 1
    read2_tags[read2.get_tag('YS')] += 1

In [11]:
read1_tags

{'C_C2T': 34927, 'W_C2T': 34655, 'W_G2A': 0, 'C_G2A': 0}

In [12]:
read2_tags

{'C_C2T': 0, 'W_C2T': 0, 'W_G2A': 34655, 'C_G2A': 34927}

# TBS

In [13]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
bam_file  = working_path + "/data/TBS/20000.sorted.mdup.bam"

In [14]:
read1_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}
read2_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}

for read1, read2 in read_pair_generator(bam_file, 'chr21'):
    read1_tags[read1.get_tag('YS')] += 1
    read2_tags[read2.get_tag('YS')] += 1

In [15]:
read1_tags

{'C_C2T': 180709, 'W_C2T': 23980, 'W_G2A': 0, 'C_G2A': 0}

In [16]:
read2_tags

{'C_C2T': 0, 'W_C2T': 0, 'W_G2A': 23980, 'C_G2A': 180709}

# Simulate

In [21]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
bam_file  = working_path + "/data/sim/pe_d/sim.mkdup.sorted.bam"

In [22]:
read1_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}
read2_tags = {'C_C2T':0 , 'W_C2T':0, 'W_G2A':0, 'C_G2A':0}

for read1, read2 in read_pair_generator(bam_file,'chr10'):
    read1_tags[read1.get_tag('YS')] += 1
    read2_tags[read2.get_tag('YS')] += 1

In [23]:
read1_tags

{'C_C2T': 14414, 'W_C2T': 14210, 'W_G2A': 0, 'C_G2A': 0}

In [24]:
read2_tags

{'C_C2T': 0, 'W_C2T': 0, 'W_G2A': 14210, 'C_G2A': 14414}